# Preparing the data

## Define parameters


In [35]:
from sklearn.svm import SVR
kernels = ['linear','‘poly', 'rbf', 'sigmoid']
C_values = [0.2, 0.4, 0.6, 0.8, 1.0]
rbf_gamma = [0.05, 0.2, 0.4, 0.6]
ps_coef0 = [0.0, 0.05, 0.1, 0.15, 0.2, 0.5]
poly_degree = [1, 2, 3, 5]
train_set_size = 5000

## Load the data

In [36]:
# Load the data

from pathlib import Path
import pandas as pd
import tarfile
import urllib.request
from sklearn.model_selection import train_test_split

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets", filter="data")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing_full = load_housing_data()


### Create train and test set

In [37]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler, StandardScaler


train_set, test_set = train_test_split(housing_full, test_size=0.2,
                                       random_state=2911)
train_set = train_set[:train_set_size]

train_set.head(8)
train_labels = train_set["median_house_value"].copy()
train_set.drop("median_house_value", axis=1)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
14333,-117.91,33.76,22.0,7531.0,1569.0,5254.0,1523.0,3.8506,<1H OCEAN
11711,-117.31,34.15,7.0,5747.0,1307.0,2578.0,1147.0,3.3281,INLAND
2424,-121.92,36.62,52.0,1410.0,303.0,578.0,285.0,2.5625,NEAR OCEAN
1972,-118.21,34.64,16.0,2573.0,427.0,1273.0,426.0,5.9508,INLAND
14697,-117.99,33.92,27.0,5805.0,1152.0,3106.0,1144.0,4.0610,<1H OCEAN
...,...,...,...,...,...,...,...,...,...
13126,-122.43,37.74,52.0,2229.0,498.0,1079.0,472.0,5.0196,NEAR BAY
6432,-121.89,37.49,9.0,4909.0,577.0,1981.0,591.0,9.7194,<1H OCEAN
181,-118.10,34.01,29.0,2077.0,564.0,2087.0,543.0,2.6600,<1H OCEAN
17835,-121.57,39.48,15.0,202.0,54.0,145.0,40.0,0.8252,INLAND


In [38]:
# cat_encoder = OneHotEncoder(sparse_output=False)
# housing_cat = train_set[["ocean_proximity"]]

# housing_cat_1hot = pd.DataFrame(cat_encoder.fit_transform(housing_cat),
#                          columns=cat_encoder.get_feature_names_out(),
#                          index=housing_cat.index)
# # print(cat_encoder.feature_names_in_)
# # print(cat_encoder.get_feature_names_out())


# # Drop the original 'ocean_proximity' column from train_set
# train_set_numerical = train_set.drop("ocean_proximity", axis=1)

In [39]:
# Concatenate the numerical features with the one-hot encoded features
# train_set = pd.concat([train_set_numerical, housing_cat_1hot], axis=1)
# print(train_set.info())
# print(train_set.describe())
# std_scaler = StandardScaler()
# std_scaler.set_output(transform="pandas")

# housing_num_std_scaled = std_scaler.fit_transform(train_set_numerical)
# housing_num_std_scaled = pd.concat([housing_num_std_scaled, housing_cat_1hot], axis = 1)
# housing_num_std_scaled.head(8)

# Prepare the models

## Define ClusterSimilarity

In [40]:
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.kernel_approximation import RBFSampler
from sklearn.metrics.pairwise import rbf_kernel
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  # always return self!

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

## Define log pipeline
log pipeline will transform the data to log, and perform a standard scaler on it.


In [41]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer


def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]  # feature names out

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler())

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler())

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"),
                                     StandardScaler())




## Perform column transformation
separate the numerical and categorical attributes, define some extra columns, like rational ones, and also cluster similarity for geographical features.

In [42]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_selector, make_column_transformer

num_attribs = ["longitude", "latitude", "housing_median_age", "total_rooms",
               "total_bedrooms", "population", "households", "median_income"]
cat_attribs = ["ocean_proximity"]

cat_pipeline = Pipeline([
    ("impute_mf", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore"))
    ])



num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])
preprocessing = ColumnTransformer([
        ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
        ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
        ("people_per_house", ratio_pipeline(), ["population", "households"]),
        ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                               "households", "median_income"]),
        ("geo", cluster_simil, ["latitude", "longitude"]),
        ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
    ],
    remainder=default_num_pipeline)

# preprocessing.set_output(transform="pandas")
housing_prepared = preprocessing.fit_transform(train_set)
housing_prepared_fr = pd.DataFrame(housing_prepared, columns=preprocessing.get_feature_names_out(),index=train_set.index)
housing_prepared_fr.columns

Index(['bedrooms__ratio', 'rooms_per_house__ratio', 'people_per_house__ratio',
       'log__total_bedrooms', 'log__total_rooms', 'log__population',
       'log__households', 'log__median_income', 'geo__Cluster 0 similarity',
       'geo__Cluster 1 similarity', 'geo__Cluster 2 similarity',
       'geo__Cluster 3 similarity', 'geo__Cluster 4 similarity',
       'geo__Cluster 5 similarity', 'geo__Cluster 6 similarity',
       'geo__Cluster 7 similarity', 'geo__Cluster 8 similarity',
       'geo__Cluster 9 similarity', 'cat__ocean_proximity_<1H OCEAN',
       'cat__ocean_proximity_INLAND', 'cat__ocean_proximity_ISLAND',
       'cat__ocean_proximity_NEAR BAY', 'cat__ocean_proximity_NEAR OCEAN',
       'remainder__housing_median_age', 'remainder__median_house_value'],
      dtype='object')

# Define pipeline


In [43]:
from sklearn.svm import SVR

def add_prefix(step_name, params: dict) -> dict:
    return {f"{step_name}__{k}": v for k, v in params.items()}




svr = SVR()
full_pipeline = Pipeline([
      ("preprocessing", preprocessing),
      ("svr", svr)
])
# full_pipeline.get_params()

# Select and train a model


In [ ]:
train_set.head(8)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
14333,-117.91,33.76,22.0,7531.0,1569.0,5254.0,1523.0,3.8506,167400.0,<1H OCEAN
11711,-117.31,34.15,7.0,5747.0,1307.0,2578.0,1147.0,3.3281,122200.0,INLAND
2424,-121.92,36.62,52.0,1410.0,303.0,578.0,285.0,2.5625,235400.0,NEAR OCEAN
1972,-118.21,34.64,16.0,2573.0,427.0,1273.0,426.0,5.9508,181100.0,INLAND
14697,-117.99,33.92,27.0,5805.0,1152.0,3106.0,1144.0,4.0610,222700.0,<1H OCEAN
4277,-122.42,37.66,36.0,725.0,121.0,335.0,140.0,4.1250,327600.0,NEAR OCEAN
17413,-118.01,33.91,32.0,2722.0,571.0,2541.0,462.0,4.2305,221400.0,<1H OCEAN
15425,-120.95,37.61,17.0,4054.0,654.0,2034.0,667.0,4.6833,142200.0,INLAND


## Perform gridsearch

In [ ]:
from sklearn.model_selection import GridSearchCV

# kernels = ['linear','‘poly', 'rbf', 'sigmoid']
# C_values = [0.2, 0.4, 0,6, 0.8, 1.0]
# rbf_gamma = [0.05, 0.2, 0.4, 0.6]
# ps_coef0 = [0.0, 0.05, 0.1, 0.15, 0.2, 0.5]
# poly_degree = [1, 2, 3, 5]


param_grid = [
   {
       "svr__kernel": ["linear"],
       "svr__C": C_values
   },
   {
      "svr__kernel": ["poly", "sigmoid"],
      "svr__coef0": ps_coef0,
      "svr__degree": poly_degree,
      "svr__C": C_values
   },
   {
       "svr__kernel": ["rbf"],
       "svr__gamma": rbf_gamma,
       "svr__C": C_values
   }
]

# param_grid = [add_prefix("svr", param) for param in base_grid]





grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(train_set, train_labels)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                                                     SimpleImputer(strategy='median')),
                                                                                    ('standardscaler',
                                                                                     StandardScaler())]),
                                                          transformers=[('bedrooms',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('functiontransformer',
                                                                                          FunctionTransformer(feature_names_out=<f...
                                       ('svr', SVR())]),
             n_jobs=-1,
             param_grid=[{'svr__C': [0.2, 0.4, 0.6, 0.8, 1.0],
                          'svr__kernel': ['linear']},
                         {'svr__C': [0.2, 0.4, 0.6, 0.8, 1.0],
                          'svr__coef0': [0.0, 0.05, 0.1, 0.15, 0.2, 0.5],
                          'svr__degree': [1, 2, 3, 5],
                          'svr__kernel': ['poly', 'sigmoid']},
                         {'svr__C': [0.2, 0.4, 0.6, 0.8, 1.0],
                          'svr__gamma': [0.05, 0.2, 0.4, 0.6],
                          'svr__kernel': ['rbf']}],
             scoring='neg_root_mean_squared_error')

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
results["rmse"] = np.sqrt(-results["mean_test_score"])

results.sort_values("rmse")[:10]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_svr__C,param_svr__kernel,param_svr__coef0,param_svr__degree,param_svr__gamma,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,rmse
4,1.450760,0.319134,0.528097,0.017791,1.0,linear,NaN,NaN,NaN,"{'svr__C': 1.0, 'svr__kernel': 'linear'}",-113697.356993,-116237.923909,-113108.097958,-114347.792953,1358.001675,1,338.153505
3,1.877805,0.290377,0.509399,0.066167,0.8,linear,NaN,NaN,NaN,"{'svr__C': 0.8, 'svr__kernel': 'linear'}",-114551.313150,-116608.684265,-113925.680763,-115028.559393,1146.138424,2,339.158605
2,2.071460,0.411351,0.607146,0.066133,0.6,linear,NaN,NaN,NaN,"{'svr__C': 0.6, 'svr__kernel': 'linear'}",-115384.387307,-117133.221143,-114765.728528,-115761.112326,1002.562259,3,340.236847
1,2.511224,0.536432,0.777422,0.058984,0.4,linear,NaN,NaN,NaN,"{'svr__C': 0.4, 'svr__kernel': 'linear'}",-116232.420431,-117788.000316,-115557.996589,-116526.139112,933.785183,4,341.359252
0,2.058806,0.516065,0.614316,0.319893,0.2,linear,NaN,NaN,NaN,"{'svr__C': 0.2, 'svr__kernel': 'linear'}",-117123.705914,-118537.045754,-116397.031905,-117352.594524,888.522112,5,342.567650
213,1.175508,0.032682,0.375175,0.005722,1.0,poly,0.10,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.1, 'svr__degre...",-117623.800923,-119026.823436,-116883.837205,-117844.820521,888.719957,6,343.285334
229,1.184647,0.029769,0.380712,0.028787,1.0,poly,0.20,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.2, 'svr__degre...",-117623.800923,-119026.823436,-116883.837205,-117844.820521,888.719957,7,343.285334
197,1.162455,0.035924,0.386780,0.005636,1.0,poly,0.00,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.0, 'svr__degre...",-117623.800923,-119026.823436,-116883.837205,-117844.820521,888.719957,8,343.285334
205,1.175600,0.031613,0.389113,0.019578,1.0,poly,0.05,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.05, 'svr__degr...",-117623.800923,-119026.823436,-116883.837205,-117844.820521,888.719957,9,343.285334
221,1.195871,0.026569,0.363539,0.009722,1.0,poly,0.15,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.15, 'svr__degr...",-117623.800924,-119026.823436,-116883.837205,-117844.820521,888.719957,10,343.285334


## Perform RnadomizedSearch
**Exercise 2**

In [55]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from scipy.stats import randint, uniform

# kernels = ['linear','‘poly', 'rbf', 'sigmoid']
# C_values = [0.2, 0.4, 0,6, 0.8, 1.0]
# rbf_gamma = [0.05, 0.2, 0.4, 0.6]
# ps_coef0 = [0.0, 0.05, 0.1, 0.15, 0.2, 0.5]
# poly_degree = [1, 2, 3, 5]
C_dist = uniform(0.0, 1.0)
ps_coef0_dist = uniform(0.0, 1.0)
rbf_gamma_dist = uniform(0.0, 1.0)
param_distrs = [
   {
       "svr__kernel": ["linear"],
       "svr__C" : C_dist
   },
   {
      "svr__kernel": ["poly", "sigmoid"],
      "svr__coef0": ps_coef0_dist,
      "svr__degree": randint(1, 5),
      "svr__C": C_dist
   },
   {
       "svr__kernel": ["rbf"],
       "svr__gamma": rbf_gamma_dist,
       "svr__C": C_dist
   }
]

svr = SVR()

random_search = RandomizedSearchCV(full_pipeline,
                                   param_distrs,
                                   cv=3,
                                   scoring="neg_root_mean_squared_error",)

In [56]:
rnd_search = random_search.fit(train_set, train_labels)

In [57]:
cv_res = pd.DataFrame(rnd_search.cv_results_)
cv_res["rmse"] = np.sqrt(-cv_res["mean_test_score"])

cv_res.sort_values("rmse")[:10]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_svr__C,param_svr__gamma,param_svr__kernel,param_svr__coef0,param_svr__degree,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,rmse
2,0.598674,0.019207,0.203190,0.002659,0.724949,NaN,linear,NaN,NaN,"{'svr__C': 0.7249485806645909, 'svr__kernel': ...",-114856.546846,-1.167857e+05,-114239.427850,-1.152939e+05,1.084537e+03,1,339.549543
1,0.611813,0.043779,0.197379,0.006374,0.232758,NaN,linear,NaN,NaN,"{'svr__C': 0.23275824560775604, 'svr__kernel':...",-116976.424588,-1.183814e+05,-116246.117413,-1.172013e+05,8.861256e+02,2,342.346790
5,0.593286,0.009628,0.205245,0.008492,0.159323,NaN,linear,NaN,NaN,"{'svr__C': 0.15932339477516455, 'svr__kernel':...",-117305.195961,-1.187118e+05,-116578.638225,-1.175319e+05,8.854948e+02,3,342.829233
4,0.773059,0.001038,0.524271,0.007883,0.760757,0.281918,rbf,NaN,NaN,"{'svr__C': 0.7607569097815234, 'svr__gamma': 0...",-117965.269414,-1.193572e+05,-117181.558720,-1.181680e+05,8.996816e+02,4,343.755724
3,1.056588,0.259851,0.744063,0.183030,0.890740,0.409003,rbf,NaN,NaN,"{'svr__C': 0.8907395420375485, 'svr__gamma': 0...",-117984.846613,-1.193735e+05,-117197.656769,-1.181853e+05,8.995117e+02,5,343.780924
6,1.640329,0.276642,0.636226,0.140880,0.025371,NaN,sigmoid,0.476941,3.0,"{'svr__C': 0.02537063239645665, 'svr__coef0': ...",-118016.661136,-1.193975e+05,-117224.309351,-1.182128e+05,8.979615e+02,6,343.820899
9,0.781327,0.014064,0.635036,0.162374,0.597896,0.784027,rbf,NaN,NaN,"{'svr__C': 0.5978956665916023, 'svr__gamma': 0...",-118016.257246,-1.193985e+05,-117224.036718,-1.182129e+05,8.985361e+02,7,343.821061
0,0.746213,0.010795,0.533709,0.026125,0.039871,0.352069,rbf,NaN,NaN,"{'svr__C': 0.03987100003506594, 'svr__gamma': ...",-118021.676494,-1.194024e+05,-117228.655306,-1.182176e+05,8.981548e+02,8,343.827810
7,0.647585,0.019472,0.235861,0.009168,0.975202,NaN,poly,0.360720,2.0,"{'svr__C': 0.9752021001101268, 'svr__coef0': 0...",-117669.839065,-2.545336e+05,-116917.463272,-1.630403e+05,6.469626e+04,9,403.782490
8,0.648227,0.014638,0.221712,0.011496,0.999525,NaN,poly,0.171148,3.0,"{'svr__C': 0.9995252277472735, 'svr__coef0': 0...",-117640.652128,-2.355308e+08,-116828.280211,-7.858844e+07,1.109750e+08,10,8865.011958


## Peform feature selection
### Exercise 3

In [ ]:
rnd_search.get_params(False)

{'cv': 3,
 'error_score': nan,
 'estimator': Pipeline(steps=[('preprocessing',
                  ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                               SimpleImputer(strategy='median')),
                                                              ('standardscaler',
                                                               StandardScaler())]),
                                    transformers=[('bedrooms',
                                                   Pipeline(steps=[('simpleimputer',
                                                                    SimpleImputer(strategy='median')),
                                                                   ('functiontransformer',
                                                                    FunctionTransformer(feature_names_out=<function ratio_name at 0x787b836...
                                                    'total_rooms', 'population',
            

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

select_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('feature_selection', SelectFromModel(RandomForestRegressor(random_state=42),
                                          threshold=0.005)),
    ('svr', SVR(C=rnd_search.best_params_["svr__C"],
                kernel=rnd_search.best_params_["svr__kernel"]))
])


In [ ]:
selector_rmses =  -cross_val_score(select_pipeline,
                                  train_set,
                                  train_labels,
                                  scoring="neg_root_mean_squared_error",
                                  cv=3)
pd.Series(selector_rmses).describe()

,0
count,3.000000
mean,115654.399596
std,1069.454143
min,114694.781664
25%,115077.938197
50%,115461.094731
75%,116134.208563
max,116807.322394


# Create a custom transformer for knn
### Exercise 4


In [59]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted
from sklearn.base import clone


class KNeighborsTransformer(BaseEstimator, TransformerMixin):

  def __init__(self, n_neighbors=3):
    self.n_neighbors = n_neighbors # Store n_neighbors as an attribute
    self.knn_regressor = KNeighborsRegressor(n_neighbors=n_neighbors)


  def fit(self, X, y=None):
    self.n_features_in_ = X.shape[1]
    self.knn_regressor.fit(X, y)
    return self


  def transform(self, X):
    check_is_fitted(self)
    check_array(X)
    assert self.n_features_in_ == X.shape[1]
    return pd.DataFrame(self.knn_regressor.predict(X), columns=["knn_prediction"])


# won't work, since it is meant for stateless transformers.
# knn_transformer = FunctionTransformer(knnr, inverse_func=knnr.inverse_transform)
# knn_loc = knn_transformer.transform(train_set[["latitude", "longitude"]])

knn_transformer = KNeighborsTransformer()
knn_loc = knn_transformer.fit_transform(train_set[["latitude", "longitude"]], train_set['median_income'])

transformers = [(name, clone(transformer), columns)
                for name, transformer, columns in preprocessing.transformers]


geo_index = [name for name, _, _ in transformers].index("geo")


transformers[geo_index] = ("geo", knn_transformer,
                           ["latitude", "longitude", "median_income"])

new_geo_preprocessing = ColumnTransformer(transformers)
geo_pipeline = Pipeline([
    ('preprocessing', new_geo_preprocessing),
    ('svr', SVR(C=rnd_search.best_params_["svr__C"],
                kernel=rnd_search.best_params_["svr__kernel"]))
])

In [61]:
from sklearn.model_selection import cross_val_score

new_pipe_rmses = -cross_val_score(geo_pipeline,
                                  train_set,
                                  train_labels,
                                  scoring="neg_root_mean_squared_error",
                                  cv=3)
pd.Series(new_pipe_rmses).describe()

,0
count,3.000000
mean,68224.843784
std,4229.397582
min,63454.582769
25%,66579.161190
50%,69703.739612
75%,70609.974291
max,71516.208970
